**pybinding of pSZ/cuSZ**: whole-pipeline compress / decompress

Navigate to the repo root: 
```sh
CUDAARCHS=120 pip install -e py_ext --no-build-isolation
```

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import cupy as cp
import numpy as np
from psz import *

print(f"cupy {cp.__version__}")
print(cp.cuda.runtime.getDeviceProperties(0)["name"].decode())

**Data**: Load data with user-interpreted shape. We use `float`-type CESM-CLDHGH, whose shape is $1800_y$-by-$3600_x$, for an example.

In [ ]:
PATH = os.environ.get("CESM", "/path/to/cesm-3600x1800.f32")
SHAPE = (1800, 3600)  # (y, x), slow to fast
EB = 1e-3  # relative to the data range

data = cp.fromfile(PATH, dtype=cp.float32).reshape(SHAPE)
print(f"{PATH}\n{SHAPE}  {data.nbytes / 2**20:.1f} MiB")

Before **compression** by calling `compress_process`, `compress_extrema` is optionally caleed to give the data range that turns the relative `EB` into an absolute bound.

For data processing by going through the compression pipeline, the pipeline can be specified when calling `compress_process`. In this case, `lrz..` is specified, which stands for Lorenzo predictor and defaults for all subsequant stages.

For **decompression**, `decompress_process` is the reverse process of `compress_process`.

For timing purposes, `compress_process` and `decompress_process` reflect the "all-kernel" process, while `compress_process` + `compress_archive` and `decompress_process` can be seen as "end-to-end" process for one-time-run purposes.

In [ ]:
with compress_init(SHAPE, cp.float32) as c:
    summary = c.compress_extrema(data)
    c.compress_process("lrz..", data, EB * summary.rng)
    header, archive = c.compress_archive()

with decompress_init(header) as dc:
    xdata = dc.decompress_process(archive)

q = assess_quality_float(xdata, data)  # evaluated on GPU
print_concise_quality(header, q, archive.nbytes)

**Reports** print from C to the kernel's stdout (the Jupyter terminal); `%load_ext wurlitzer` shows them here.

In [ ]:
review_compression(header)
review_decompression(header)
review_compression(header, verbose=True)

**Comparing pipelines** in one 3-stage compressor, switched per `compress_process`; the last three are the two-pass `hicr`, `hitp`, `hitp_r1` presets.

In [ ]:
PIPELINES = ["lrz..", "lrz,_", "spl,_", "lrz-zz,fzg", "spl,hf,lc-rtr", "spl,lc-tcms,lc-bitr", "spl,lc-drh,lc-bitr"]
out = cp.empty(SHAPE, dtype=cp.float32)

def run(c, pipeline):
    c.compress_reset()
    c.compress_process(pipeline, data, EB * summary.rng)
    header, archive = c.compress_archive()
    with decompress_init(header) as dc:
        xdata = dc.decompress_process(archive, out=out)
    q = assess_quality_float(xdata, data)
    return {"pipeline": pipeline, "ratio": data.nbytes / archive.nbytes, "psnr": q.psnr, "max_err": q.max_err_abs}


with compress_init_3stage(SHAPE, cp.float32) as c:
    rows = [run(c, p) for p in PIPELINES]

print(f"{'pipeline':22s} {'ratio':>8s} {'PSNR':>8s} {'max err':>11s}")
for row in rows:
    print(f"{row['pipeline']:22s} {row['ratio']:7.2f}x {row['psnr']:7.1f}  {row['max_err']:11.3e}")

In [ ]:
import shutil

import matplotlib as mpl
import matplotlib.pyplot as plt

# real LaTeX when installed, matplotlib mathtext otherwise
USE_TEX = all(shutil.which(b) for b in ("latex", "dvipng", "gs"))
mpl.rcParams["text.usetex"] = USE_TEX
mpl.rcParams["font.family"] = "Helvetica" if USE_TEX else "sans-serif"
mpl.rcParams["figure.dpi"] = 144
if USE_TEX:
    mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"
print("LaTeX:", "on" if USE_TEX else "off (latex/dvipng/gs not found; using mathtext)")


def tex(s):
    """A literal pipeline spec: monospace, with _ escaped so TeX does not subscript."""
    return rf"\texttt{{{s.replace('_', r'\_')}}}" if USE_TEX else s

In [ ]:
err = cp.asnumpy(cp.abs(xdata - data))

fig, ax = plt.subplots(figsize=(6, 6 * err.shape[0] / err.shape[1]))
ax.imshow(err, cmap="Blues", vmin=0, vmax=err.max(), interpolation="nearest")
ax.set_title(rf"Pointwise $|$error$|$, Reconstructed from {tex('lrz,_')} Pipeline")
ax.set_xticks([])
ax.set_yticks([])
for s in ax.spines.values():
    s.set_visible(False)
fig.tight_layout()